<div style='text-align: center; padding: 30px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); border-radius: 15px; margin: 10px 0; box-shadow: 0 10px 30px rgba(0,0,0,0.2);'>
  <h1 style='color: white; margin: 0 0 8px 0; font-size: 2.5em;'>🎵 ACE-Step 1.5 AI Music Generator</h1>
  <h3 style='color: #f0f0f0; margin: 0 0 5px 0; font-weight: 400;'>Custom UI Edition — Created by <strong>AIQUEST</strong></h3>
  <p style='color: #ddd; margin: 0 0 0 0;'>Free on Google Colab T4 GPU | MIT License</p>
</div>

---

<div align="center">

  <img src="https://img.shields.io/badge/AIQUESTAcademy-blueviolet?style=for-the-badge&logo=youtube&logoColor=white" />
  <img src="https://img.shields.io/badge/Colab-Free%20T4%20GPU-4285F4?style=for-the-badge&logo=google-colab&logoColor=white" />

  <br>

  <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1">
    <img src="https://img.shields.io/badge/Subscribe%20on%20YouTube-FF0000?style=for-the-badge&logo=youtube&logoColor=white" />
  </a>
  <a href="https://x.com/aiquestacademy">
    <img src="https://img.shields.io/badge/Follow%20on%20X-000000?style=for-the-badge&logo=x&logoColor=white" />
  </a>

</div>

---


### What is ACE-Step 1.5?

**ACE-Step 1.5** is a powerful open-source music generation model that creates complete songs from text prompts and lyrics.

| Feature | Details |
|---------|----------|
| Speed | Generate music in seconds |
| Languages | 50+ languages supported |
| Modes | Text2Music, Cover, Repaint |
| License | MIT - Free for commercial use |
| Control | BPM, key, duration, time signature |

### Before You Start
> Go to **Runtime -> Change runtime type -> T4 GPU**

---
## Step 1: Verify GPU & Install PyTorch

Checks your GPU and installs PyTorch 2.3.1 to avoid CUDA OOM errors.

> If no GPU is detected -> **Runtime -> Change runtime type -> T4 GPU**


In [ ]:
# @title Step 1: Verify GPU
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
print()

import torch
print(f"  PyTorch {torch.__version__}")
print(f"  CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

In [ ]:
# @title Step 2: Install uv
import sys
print(f"Python: {sys.version}")
assert sys.version_info >= (3, 11), "ACE-Step 1.5 requires Python >= 3.11"

print("\nInstalling uv package manager...")
!curl -LsSf https://astral.sh/uv/install.sh | sh 2>&1 | tail -1

import os
os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]
!uv --version
print("Done!")

In [ ]:
# @title Step 3: Clone & Install
%cd /content

import os
if os.path.exists("/content/ACE-Step-1.5"):
    print("Repository exists, pulling latest...")
    %cd /content/ACE-Step-1.5
    !git pull
else:
    print("Cloning ACE-Step 1.5...")
    !git clone https://github.com/ACE-Step/ACE-Step-1.5.git
    %cd /content/ACE-Step-1.5

print("\nInstalling dependencies (this may take 2-3 minutes)...\n")
!uv sync
print("\nAll dependencies installed!")

In [ ]:
# @title Step 4: Launch Music Generator
%cd /content/ACE-Step-1.5

import os
os.environ["TORCH_CUDNN_SDPA_ENABLED"] = "1"
os.environ["ATTN_BACKEND"] = "sdpa"
os.environ["MPLBACKEND"] = "agg"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]

LAUNCHER = '''
import sys, os, importlib, pkgutil, traceback
os.environ["MPLBACKEND"] = "agg"

def run_custom_ui(dit, llm, **kwargs):
    print("🎨 Launching Custom Music Studio UI...")
    import gradio as gr
    import random
    import torch
    from acestep.inference import generate_music, GenerationParams, GenerationConfig, format_sample

    try:
        from acestep.gradio_ui.ui_components import EXAMPLE_POOL
        print(f"✅ Loaded {len(EXAMPLE_POOL)} examples from original UI")
    except:
        try:
            from acestep.gradio_ui.examples import get_examples
            EXAMPLE_POOL = get_examples()
            print(f"✅ Loaded {len(EXAMPLE_POOL)} examples")
        except:
            EXAMPLE_POOL = [
                {"caption": "aggressive, high-energy heavy metal, dual heavily distorted guitars, chugging riffs, powerful driving drums, male vocals forceful raspy, melodic technical guitar solo, fast runs expressive bends, clean powerful production, anthemic rebellious",
                 "lyrics": "[Intro - Guitar Riff]\\n\\n[Verse 1]\\nRip the script, break it mold\\nShatter chains, dare the bold\\nTook your crown, you can\'t ignore\\nFreedom\'s knock on every door\\n\\n[Chorus]\\nBreak the chains, let it pour\\nShout it loud, beg no more\\nBurn the rules, end the war\\nWe\'re alive, can\'t ignore\\nCan\'t ignore\\n\\n[Verse 2]\\nTear it down, brick by brick\\nEvery lie, every trick\\nSkies on fire, won\'t obey\\nChaos calls, don\'t delay\\n\\n[Chorus]\\nBreak the chains, let it pour\\nShout it loud, beg no more\\nBurn the rules, end the war\\nWe\'re alive, can\'t ignore\\nCan\'t ignore\\n\\n[Guitar Solo]\\n\\n[Verse 3]\\nEchoes scream, hearts collide\\nNo surrender, no divide\\nWolves are howling, hear their cry\\nWe rise up, won\'t comply\\n\\n[Chorus]\\nBreak the chains, let it pour\\nShout it loud, beg no more\\nBurn the rules, end the war\\nWe\'re alive, can\'t ignore\\nCan\'t ignore\\n\\n[Outro - Guitar Riff]\\n[abrupt silence]",
                 "language": "en"},
                {"caption": "female vocals, rap, modern, hip hop, Indian fusion, whispered, plucked synth melody, drums of earth",
                 "lyrics": "[Intro - Plucked Synth Melody]\\n\\n[Verse 1]\\nThe sun melts over the endless plain\\nA mirage dances like a golden chain\\nBare feet trace stories in the cracked dry land\\nWho holds the map in this empty game?\\n\\n[Chorus]\\nDesert echoes\\nThey call my name\\nDrums of the earth, they beat the same\\nStep to the rhythm, no one to blame\\nDesert echoes\\nDesert echoes\\n\\n[Instrumental Break - Plucked Synth Melody]\\n\\n[Verse 2]\\nA lone wind whispers secrets untold\\nStars blink brighter than the city\'s gold\\nMy shadow stretches long in the moonlit fold\\nThis is my world, fierce and bold\\n\\n[Bridge]\\nDo you hear it? The silence screams\\nDid you see it? The shifting dreams\\n[Vocal chop: \'dreams\']\\n\\n[Chorus]\\nDesert echoes\\nThey call my name\\nDrums of the earth, they beat the same\\nStep to the rhythm, no one to blame\\nDesert echoes\\n\\n[Plucked synth melody fades out]",
                 "language": "en"},
                {"caption": "romantic active synthpop anthem, joyful energy, irresistible charm, bright engaging female vocals, feelgood danceable",
                 "lyrics": "Midnight talks with my regrets,\\nWhiskey lips and cigarettes.\\nPromises like shattered glass,\\nCut me deep but never last.\\n\\nI danced with fire, played the fool,\\nBroke the rules and called it cool.\\nNow the mirror wont lie\\nI dont recognize these eyes.\\n\\nI was right, I was left\\nCaught between the lines, no regrets\\nI was wrong, I was left behind\\nLike a shadow fading with the light\\n\\nI\'ve been up, but mostly down\\nChasing dreams that hit the ground\\nEvery high just slips away\\nLeaving echoes of my name\\n\\nI was right, I was left,\\nTorn in two with every step.\\nI was wrong, I was left behind,\\nAnother love I couldnt find.\\n\\nIve been up, but mostly down,\\nChasing highs that let me drown.\\nI was right, I was left,\\nNow Im at a new low.",
                 "language": "en"},
                {"caption": "smooth jazzy lo-fi hip-hop, gentle piano melody, relaxed steady drum machine, warm round bassline, duet clear melodic female and smooth conversational male vocals, tasteful melodic saxophone fills, late-night atmosphere, extended instrumental outro, expressive improvisational sax solo",
                 "lyrics": "[Intro: Piano & Saxophone]\\n\\n[Verse 1: Female Vocal]\\n\\u6708\\u5149\\u843d\\u8fdb\\u7a97\\n\\u50cf\\u662f\\u8d77\\u8272\\u7684\\u90a3\\u676f\\n\\u4f60\\u8fd8\\u5728\\u6211\\u7684\\u5fc3\\u5e95\\u65cb\\u8f6c\\n\\u629b\\u5f00\\u6240\\u6709\\u70e6\\u5fe7\\n\\u8fd9\\u7231\\u50cf\\u6d41\\u661f\\u5760\\u843d\\n\\u5728\\u591c\\u7a7a\\u95ea\\u8000\\n\\u65e0\\u9700\\u89e3\\u7b54 \\u5fc3\\u52a8\\u662f\\u552f\\u4e00\\u7684\\u89e3\\u836f\\n\\n[Verse 2: Male Vocal]\\n\\u4f60\\u7684\\u773c\\u795e\\u50cf\\u8ff7\\u5bab\\n\\u6211\\u5f98\\u5f8a\\u5176\\u4e2d\\n\\u6bcf\\u4e2a\\u97f3\\u7b26\\u90fd\\u5e26\\u4e86\\u5fc3\\u810f\\u7684\\u8f70\\u9e23\\n\\u89e6\\u78b0\\u4f60\\u7684\\u6e29\\u5ea6\\u5c31\\u50cf\\u7535\\u6d41\\u7a7f\\u5fc3\\n\\u65e0\\u6cd5\\u6297\\u62d2\\u4f60\\u7684\\u7231\\u628a\\u6211\\u62c9\\u5165\\u68a6\\u5883\\n\\n[Chorus: Duet]\\n\\u5fc3\\u8df3\\u968f\\u7740\\u4f60\\u72c2\\u5954\\n\\u5982\\u6d77\\u6f6e\\u6c39\\u6d8c\\n\\u7231\\u7684\\u8282\\u594f\\u7ffb\\u6eda\\u7740\\n\\u50cf\\u68a6\\u5883\\u4e2d\\u68a6\\n\\u4f60\\u7684\\u540d\\u5b57\\u662f\\u65cb\\u5f8b\\n\\u65cb\\u5f8b\\u4e2d\\u6210\\u98ce\\n\\u6bcf\\u4e00\\u62cd\\u90fd\\u8bc9\\u8bf4\\u7231\\u65e0\\u58f0\\u7684\\u60b8\\u52a8\\n\\n[Instrumental Break: Saxophone Solo]\\n\\n[Verse 3: Male Vocal]\\n\\u9ed1\\u591c\\u50cf\\u4e2a\\u821e\\u53f0\\n\\u6211\\u4eec\\u70b9\\u4e86\\u5f69\\u706f\\n\\u8282\\u62cd\\u8ddf\\u968f\\u5fc3\\u8df3\\n\\u9ed8\\u5951\\u878d\\u6210\\u4e00\\u543b\\n\\u522b\\u8ba9\\u591c\\u665a\\u505c\\u4e0b\\u6765\\n\\u7231\\u6c38\\u8fdc\\u5728\\u8ba4\\u771f\\n\\u6bcf\\u4e00\\u79d2\\u90fd\\u50cf\\u6b4c\\n\\u5f00\\u542f\\u5947\\u5999\\u4eba\\u751f\\n\\n[Chorus: Duet]\\n\\u5fc3\\u8df3\\u968f\\u7740\\u4f60\\u72c2\\u5954\\n\\u5982\\u6d77\\u6f6e\\u6c39\\u6d8c\\n\\u7231\\u7684\\u8282\\u594f\\u7ffb\\u6eda\\u7740\\n\\u50cf\\u68a6\\u5883\\u4e2d\\u68a6\\n\\u4f60\\u7684\\u540d\\u5b57\\u662f\\u65cb\\u5f8b\\n\\u65cb\\u5f8b\\u4e2d\\u6210\\u98ce\\n\\u6bcf\\u4e00\\u62cd\\u90fd\\u8bc9\\u8bf4\\u7231\\u65e0\\u58f0\\u7684\\u60b8\\u52a8\\n\\n[Outro: Duet & Saxophone]\\n[Saxophone melody fades out]",
                 "language": "zh"},
                {"caption": "explosive high-energy K-pop EDM, relentless four-on-the-floor beat, pulsing synth bassline, bright piano melody, shimmering arpeggiated synths, powerful clear male lead vocal, anthemic melody, energetic ad-libs hype-man shouts",
                 "lyrics": "[Intro]\\n[Synth arpeggio and pads]\\n\\n[Verse 1]\\n[ko] hwangholhan i jilseo-e neon ppajyeo\\n[en] (oh yeah)\\n[ko] lideum wi-e sinhoneul jilleo\\n[en] (oh, let\'s go)\\n\\n[Chorus]\\n[ko] oelo-un i jilseo gip-eojyeo\\n[en] (yeah, yeah)\\n[ko] lideum wi-e him-eul jwi-yeo\\n[en] (let\'s go)\\n\\n[Instrumental Break]\\n[Synth lead melody with vocal chops]\\n\\n[Outro]\\n[abrupt silence]",
                 "language": "ko"},
                {"caption": "rap, minimal, spoken word, dark, minimal beats heavy basslines, sharp snares, female vocals, introspective raw energy",
                 "lyrics": "[Intro]\\n[Music box synth]\\n[whispered] Yeah\\n\\n[Verse 1]\\n\\u9019\\u4e0d\\u662f\\u6211\\u7684\\u8857 \\u6211\\u7684\\u6839\\n\\u5730\\u4e0b\\u7684 pum pum \\u50cf\\u5fc3\\u8df3\\n\\u6572\\u97ff\\u90a3\\u5f8b\\u52d5 \\u6211\\u7684\\u5144\\u5f1f\\n\\n[Chorus]\\n\\u4e0d\\u662f\\u62b1\\u6028 \\u53ea\\u662f\\u770b\\u6e05\\n\\u5728\\u9019\\u8ff7\\u5e7b\\u7684\\u591c\\u88e1\\u601d\\u8003\\u4eba\\u751f\\u7684\\u610f\\u7fa9\\n\\n[Outro]\\n[Beat fades]",
                 "language": "zh"},
                {"caption": "driving post-punk, layered electric guitars, male lead vocal angsty strained, anthemic shouted chorus",
                 "lyrics": "[Intro - Guitar]\\n\\n[Verse 1]\\nUnder neon lights, they flicker and fade\\nLost in the hum of this restless parade\\n\\n[Chorus]\\nOh, shadows of neon\\nWhere do you run?\\n\\n[Guitar Solo]\\n\\n[Outro - fade]",
                 "language": "en"},
                {"caption": "classic country-folk ballad, steady acoustic guitar, warm male vocal gentle twang, plaintive harmonica",
                 "lyrics": "[Intro - Acoustic Guitar]\\n\\n[Verse 1]\\nFound you standing in the river\'s roar\\nHeartbeats racing like never before\\n\\n[Chorus]\\nI\'ll ride these heartstrings all night long\\nSing our song where we belong\\n\\n[Outro - Harmonica]",
                 "language": "en"},
                {"caption": "Funk, Soul, Groove, Male Vocals, Slap Bass, Horn Section, 1970s, Party Energy",
                 "lyrics": "[Drum Break]\\nGet on up!\\n\\n[Verse 1]\\nI feel the rhythm moving in my feet\\n\\n[Chorus]\\nGive it up, turn it loose!\\nWe got the funk!",
                 "language": "en"},
                {"caption": "symphonic metal epic duet, female soprano male baritone, orchestral strings, operatic choirs",
                 "lyrics": "[Intro - Orchestral]\\n\\n[Verse 1 - Female]\\nI see you walking through the ash\\n\\n[Chorus - Duet]\\nWe stand upon the edge of the night\\n\\n[Outro - fade to black]",
                 "language": "en"},
            ]
            print(f"✅ Loaded {len(EXAMPLE_POOL)} custom examples")

    LANG = {
        "en": "🇬🇧 English", "zh": "🇨🇳 Chinese", "ja": "🇯🇵 Japanese", "ko": "🇰🇷 Korean", "es": "🇪🇸 Spanish",
        "fr": "🇫🇷 French", "de": "🇩🇪 German", "it": "🇮🇹 Italian", "pt": "🇵🇹 Portuguese", "ru": "🇷🇺 Russian",
        "ar": "🇸🇦 Arabic", "hi": "🇮🇳 Hindi", "tr": "🇹🇷 Turkish", "pl": "🇵🇱 Polish", "nl": "🇳🇱 Dutch",
        "sv": "🇸🇪 Swedish", "no": "🇳🇴 Norwegian", "da": "🇩🇰 Danish", "fi": "🇫🇮 Finnish", "el": "🇬🇷 Greek",
        "he": "🇮🇱 Hebrew", "th": "🇹🇭 Thai", "vi": "🇻🇳 Vietnamese", "id": "🇮🇩 Indonesian", "ms": "🇲🇾 Malay",
        "tl": "🇵🇭 Tagalog", "uk": "🇺🇦 Ukrainian", "cs": "🇨🇿 Czech", "hu": "🇭🇺 Hungarian", "ro": "🇷🇴 Romanian",
        "bg": "🇧🇬 Bulgarian", "hr": "🇭🇷 Croatian", "sr": "🇷🇸 Serbian", "sk": "🇸🇰 Slovak", "sl": "🇸🇮 Slovenian",
        "lt": "🇱🇹 Lithuanian", "lv": "🇱🇻 Latvian", "et": "🇪🇪 Estonian", "fa": "🇮🇷 Persian", "ur": "🇵🇰 Urdu",
        "bn": "🇧🇩 Bengali", "ta": "🇮🇳 Tamil", "te": "🇮🇳 Telugu", "mr": "🇮🇳 Marathi", "gu": "🇮🇳 Gujarati",
        "kn": "🇮🇳 Kannada", "ml": "🇮🇳 Malayalam", "pa": "🇮🇳 Punjabi", "sw": "🇰🇪 Swahili", "af": "🇿🇦 Afrikaans"
    }

    def lc(l): return {v: k for k, v in LANG.items()}.get(l, "en")

    def detect_language(lyrics):
        if not lyrics or lyrics.strip() == "[inst]": return "en"
        text = lyrics.lower()
        if any(char in text for char in ['\\u661f', '\\u6708', '\\u7231', '\\u68a6', '\\u5fc3']): return "zh"
        if any(char in text for char in ['\\u306e', '\\u3092', '\\u306b', '\\u306f', '\\u304c']): return "ja"
        if any(char in text for char in ['\\uc774', '\\uadf8', '\\uc800', '\\ub97c', '\\uc740']): return "ko"
        if any(word in text for word in ['esta', 'amor', 'noche']): return "es"
        if any(char in text for char in ['\\u062a', '\\u0646', '\\u0645', '\\u0644', '\\u0639']): return "ar"
        if any(char in text for char in ['\\u0e09', '\\u0e40', '\\u0e32', '\\u0e21', '\\u0e01']): return "th"
        if any(word in text for word in ['eg', 'og', 'g\\u00e5r']): return "no"
        return "en"

    PROJECT_ROOT = "/content/ACE-Step-1.5"
    OUTPUT_DIR = os.path.join(PROJECT_ROOT, "gradio_outputs")
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    def wrap_init_engine(p=gr.Progress()):
        p(0.1, desc="Checking GPU...")
        print("🚀 Initializing Engine...")
        init_kwargs = {
            "project_root": PROJECT_ROOT,
            "config_path": "acestep-v15-turbo",
            "device": "cuda",
            "offload_to_cpu": True,
            "compile_model": False,
            "use_flash_attention": False
        }
        try:
            p(0.3, desc="Loading DiT...")
            status, ok = dit.initialize_service(**init_kwargs)
            if not ok:
                raise RuntimeError(f"DiT Init Failed: {status}")

            # ============================================================
            # FIX: Force float32 on pre-Ampere GPUs (T4 = compute 7.5)
            # The codebase sets dtype=float16 for pre-Ampere but the
            # diffusion sampling loop overflows fp16 range (max 65504),
            # producing NaN latents. ACESTEP_DTYPE env var is NOT
            # implemented — only ACESTEP_ROCM_DTYPE exists for ROCm.
            # We must directly patch the handler after init.
            # ============================================================
            if torch.cuda.is_available():
                major, _ = torch.cuda.get_device_capability()
                if major < 8:  # pre-Ampere (T4, P100, V100, etc.)
                    print("⚠️ Pre-Ampere GPU detected — forcing float32 for DiT to prevent NaN")
                    dit.dtype = torch.float32
                    if hasattr(dit, 'model') and dit.model is not None:
                        dit.model = dit.model.float()
                        print(f"   ✅ DiT model cast to float32")
                    if hasattr(dit, 'silence_latent') and dit.silence_latent is not None:
                        dit.silence_latent = dit.silence_latent.float()
                        print(f"   ✅ silence_latent cast to float32")
                    # Also cast text_encoder and vae dtype references
                    print(f"   ✅ dit.dtype = {dit.dtype}")

            print("✅ DiT loaded")
        except Exception as e:
            raise gr.Error(f"Failed to initialize DiT: {e}")
        p(0.5, desc="Checking LLM model...")
        lm_model_name = "acestep-v15-xl-sft"
        lm_checkpoint_dir = os.path.join(PROJECT_ROOT, "checkpoints")
        lm_model_dir = os.path.join(lm_checkpoint_dir, lm_model_name)
        lm_loaded = False

        # Auto-download 0.6B model from HuggingFace if not present
        if not os.path.exists(lm_model_dir):
            print(f"📥 Downloading {lm_model_name} from HuggingFace...")
            p(0.55, desc=f"Downloading {lm_model_name}...")
            try:
                from huggingface_hub import snapshot_download
                snapshot_download(
                    repo_id=f"ACE-Step/{lm_model_name}",
                    local_dir=lm_model_dir,
                    local_dir_use_symlinks=False,
                )
                print(f"✅ Downloaded {lm_model_name}")
            except Exception as dl_e:
                print(f"⚠️ Failed to download LLM: {dl_e}")

        p(0.7, desc="Loading LLM (0.6B via PyTorch)...")
        try:
            lm_status, lm_ok = llm.initialize(
                checkpoint_dir=lm_checkpoint_dir,
                lm_model_path=lm_model_name,
                backend="pt",
                device="cuda",
                offload_to_cpu=True
            )
            if lm_ok:
                lm_loaded = True
                print("✅ LLM loaded successfully!")
            else:
                print(f"⚠️ LLM init returned: {lm_status}")
        except Exception as e:
            print(f"⚠️ LLM failed: {e}")
            traceback.print_exc()

        p(1.0, desc="Ready!")
        suffix = " + LLM (0.6B CoT)" if lm_loaded else " (DiT only, no LLM)"
        return f"✅ Engine Ready!{suffix}", gr.update(interactive=True), gr.update(visible=False)

    def randomize_example():
        example = random.choice(EXAMPLE_POOL)
        caption = example.get("caption", "")
        lyrics = example.get("lyrics", "[inst]")
        lang_code = example.get("language", detect_language(lyrics))
        lang_display = LANG.get(lang_code, LANG["en"])
        print(f"🎲 Random: {caption[:50]}... | Language: {lang_display}")
        return caption, lyrics, lang_display

    def gen_music_ui(tags, lyrics, dur, lang, ntracks, think, seed, bpm, key, ts, steps, guide, instrumental, p=gr.Progress()):
        if not tags: raise gr.Error("Style/Tags required!")
        final_lyrics = "[inst]" if instrumental else (lyrics or "[inst]")
        s, r = (-1, True)
        if seed and str(seed).strip() not in ["", "-1"]:
            try: s, r = int(seed), False
            except: pass
        params = GenerationParams(caption=tags, lyrics=final_lyrics, thinking=think, duration=float(dur),
                    vocal_language=lc(lang), inference_steps=int(steps), guidance_scale=float(guide),
                    seed=s, bpm=int(bpm) if bpm else None, keyscale=key, timesignature=ts, task_type="text2music")
        config = GenerationConfig(batch_size=int(ntracks), use_random_seed=r, audio_format="wav")
        p(0.2, desc="Generating...")
        res = generate_music(dit_handler=dit, llm_handler=llm, params=params, config=config, save_dir=OUTPUT_DIR)
        if not res.success: raise gr.Error(res.error)
        paths = [a.get("path") for a in res.audios if isinstance(a, dict) and a.get("path")]
        return (paths[0] if paths else None, paths[1] if len(paths) > 1 else None, f"Generated {len(paths)} track(s)")

    def gen_cover_ui(ref, tags, lyr, strength, dur, lang, n, p=gr.Progress()):
        if not ref: raise gr.Error("Upload audio!")
        params = GenerationParams(caption=tags, lyrics=lyr or "[inst]", thinking=True, duration=float(dur),
                    vocal_language=lc(lang), task_type="cover", reference_audio=ref, audio_cover_strength=float(strength))
        config = GenerationConfig(batch_size=int(n), audio_format="mp3")
        p(0.2, desc="Generating cover...")
        res = generate_music(dit_handler=dit, llm_handler=llm, params=params, config=config, save_dir=OUTPUT_DIR)
        if not res.success: raise gr.Error(res.error)
        paths = [a.get("path") for a in res.audios if isinstance(a, dict) and a.get("path")]
        return (paths[0] if paths else None, paths[1] if len(paths) > 1 else None, f"Generated {len(paths)} cover(s)")

    def gen_repaint_ui(src, tags, lyr, start, end, strength, dur, lang, p=gr.Progress()):
        if not src: raise gr.Error("Upload audio!")
        params = GenerationParams(caption=tags, lyrics=lyr or "[inst]", thinking=True, duration=float(dur),
                    vocal_language=lc(lang), task_type="repaint", src_audio=src,
                    repainting_start=float(start), repainting_end=float(end), audio_cover_strength=float(strength))
        config = GenerationConfig(batch_size=1, audio_format="mp3")
        p(0.2, desc="Repainting...")
        res = generate_music(dit_handler=dit, llm_handler=llm, params=params, config=config, save_dir=OUTPUT_DIR)
        if not res.success: raise gr.Error(res.error)
        paths = [a.get("path") for a in res.audios if isinstance(a, dict) and a.get("path")]
        return (paths[0] if paths else None, "Repaint complete")

    def format_ui(cap, lyr):
        res = format_sample(llm_handler=llm, caption=cap, lyrics=lyr)
        if not res.success: raise gr.Error(res.error)
        return (res.caption, res.lyrics, f"BPM: {res.bpm} Key: {res.keyscale}")

    CSS = """
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;600;700&display=swap');
* { font-family: 'Inter', sans-serif !important; }
.gradio-container { max-width: 1000px !important; margin: auto !important; }
.brand-header { text-align: center; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 28px; border-radius: 15px; margin-bottom: 20px; box-shadow: 0 10px 25px rgba(102,126,234,0.3); }
.brand-title { color: white; font-size: 2em; font-weight: 700; margin: 0 0 6px 0; }
.brand-subtitle { color: rgba(255,255,255,0.88); font-size: 1em; margin-bottom: 16px; }
.btn-row { display: flex; justify-content: center; gap: 10px; flex-wrap: wrap; }
.social-btn { display: inline-flex; align-items: center; justify-content: center; min-width: 150px; padding: 10px 18px; border-radius: 10px; font-weight: 700; font-size: 13px; text-decoration: none; color: white; white-space: nowrap; }
.yt-btn  { background: #FF0000; box-shadow: 0 4px 12px rgba(255,0,0,0.3); }
.x-btn   { background: #000000; box-shadow: 0 4px 12px rgba(0,0,0,0.25); }
.sup-btn { background: linear-gradient(135deg,#f6d365,#fda085); box-shadow: 0 4px 12px rgba(253,160,133,0.35); }
button.primary { background: linear-gradient(135deg, #667eea 0%, #764ba2 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; }
"""

    with gr.Blocks(css=CSS, theme=gr.themes.Soft(), title="ACE-Step 1.5 Music Studio | AIQUEST") as demo:
        BRAND_HTML = """
<div class="brand-header">
  <div class="brand-title">🎵 ACE-Step 1.5 Music Studio</div>
  <div class="brand-subtitle">Created by <strong>AIQuest Academy</strong> &nbsp;|&nbsp; AI-Powered Music Creation</div>
  <div class="btn-row">
    <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1" target="_blank" class="social-btn yt-btn">▶ Subscribe</a>
    <a href="https://x.com/aiquestacademy" target="_blank" class="social-btn x-btn">𝕏 Follow on X</a>
    <a href="https://aiquest.site" target="_blank" class="social-btn sup-btn">❤️ Support My Work</a>
  </div>
</div>
"""
        gr.HTML(BRAND_HTML)

        init_status = gr.Textbox(value="⚠️ Engine Not Started. Click Initialize below.", label="System Status", interactive=False)
        init_btn = gr.Button("🚀 Initialize Engine (Required First Step)", variant="primary", size="lg")

        with gr.Group(visible=True) as main_ui:
            with gr.Tabs():
                with gr.Tab("🎵 Create"):
                    tags = gr.Textbox(label="🎨 Music Style & Tags", placeholder="e.g., upbeat pop, electronic, female vocals...", lines=2)
                    with gr.Row():
                        lang = gr.Dropdown(list(LANG.values()), value="🇬🇧 English", label="🌍 Language (50+ supported)")
                        instrumental = gr.Checkbox(False, label="🎹 Instrumental Only")
                    lyr = gr.Textbox(label="📝 Lyrics", lines=6, placeholder="[verse]\\nYour lyrics...\\n\\nClick 🎲 for random example", submit_btn="🎲")
                    with gr.Row():
                        dur = gr.Slider(minimum=10, maximum=480, step=10, value=60, label="⏱️ Duration (sec)")
                        ntracks = gr.Slider(minimum=1, maximum=4, step=1, value=2, label="🎚️ Tracks")
                    seed = gr.Number(value=-1, label="🎲 Seed (-1 = random, each track unique)", precision=0)
                    with gr.Accordion("⚙️ Advanced Settings", open=False):
                        with gr.Row():
                            bpm = gr.Number(0, label="🥁 BPM (0 = auto)", precision=0)
                            key = gr.Dropdown(["", "C major", "D major", "E major", "F major", "G major", "A major", "B major", "C minor", "D minor", "E minor"], label="🎹 Key", value="")
                            ts = gr.Dropdown(["", "4/4", "3/4", "6/8"], label="⏰ Time Signature", value="")
                        with gr.Row():
                            steps = gr.Slider(minimum=4, maximum=50, step=1, value=30, label="🔄 Steps")
                            guide = gr.Slider(minimum=1, maximum=15, step=0.5, value=7.0, label="🎯 Guidance")
                        think = gr.Checkbox(True, label="💭 Thinking Mode")
                    btn = gr.Button("🎵 Generate Music", variant="primary", size="lg", interactive=False)
                    with gr.Row():
                        out1 = gr.Audio(label="🎧 Track 1")
                        out2 = gr.Audio(label="🎧 Track 2")
                    out_info = gr.Textbox(label="ℹ️ Generation Info", interactive=False)
                    lyr.submit(randomize_example, outputs=[tags, lyr, lang])
                    btn.click(gen_music_ui, [tags, lyr, dur, lang, ntracks, think, seed, bpm, key, ts, steps, guide, instrumental], [out1, out2, out_info])

                with gr.Tab("🎸 Cover / Remix"):
                    gr.Markdown("### Upload audio to transform into a new style")
                    ref = gr.Audio(label="🎵 Reference Audio", type="filepath")
                    with gr.Row():
                        ctags = gr.Textbox(label="🎨 Target Style", placeholder="rock, energetic, electric guitar...")
                        cstr = gr.Slider(minimum=0, maximum=1, step=0.05, value=0.5, label="🎚️ Cover Strength")
                    clyr = gr.Textbox(label="📝 New Lyrics (Optional)", lines=4, placeholder="Leave empty to keep original")
                    cbtn = gr.Button("🎸 Generate Cover", variant="primary", size="lg", interactive=False)
                    with gr.Row():
                        cout1 = gr.Audio(label="🎧 Cover 1")
                        cout2 = gr.Audio(label="🎧 Cover 2")
                    cout_info = gr.Textbox(label="ℹ️ Info", interactive=False)
                    cbtn.click(gen_cover_ui, [ref, ctags, clyr, cstr, dur, lang, ntracks], [cout1, cout2, cout_info])

                with gr.Tab("🎨 Repaint"):
                    gr.Markdown("### Regenerate a specific section")
                    rsrc = gr.Audio(label="🎵 Source Audio", type="filepath")
                    rtags = gr.Textbox(label="🎨 Repaint Style", placeholder="jazz, smooth, saxophone...")
                    rlyr = gr.Textbox(label="📝 Section Lyrics", lines=3)
                    with gr.Row():
                        rstart = gr.Number(0, label="⏮️ Start (seconds)", precision=1)
                        rend = gr.Number(-1, label="⏭️ End (-1 = end)", precision=1)
                        rstr = gr.Slider(minimum=0, maximum=1, step=0.05, value=0.7, label="🎚️ Strength")
                    rbtn = gr.Button("🎨 Repaint Region", variant="primary", size="lg", interactive=False)
                    rout_audio = gr.Audio(label="🎧 Result")
                    rout_info = gr.Textbox(label="ℹ️ Info", interactive=False)
                    rbtn.click(gen_repaint_ui, [rsrc, rtags, rlyr, rstart, rend, rstr, dur, lang], [rout_audio, rout_info])

                with gr.Tab("📝 Format Lyrics"):
                    gr.Markdown("### Auto-format lyrics and detect metadata")
                    fcap = gr.Textbox(label="🎨 Style", placeholder="pop, upbeat...")
                    flyr = gr.Textbox(label="📝 Raw Lyrics", lines=6, placeholder="Your unformatted lyrics...")
                    fbtn = gr.Button("✨ Format & Enhance", variant="primary", size="lg", interactive=False)
                    fcap_out = gr.Textbox(label="✨ Enhanced Style", interactive=False)
                    flyr_out = gr.Textbox(label="📝 Formatted Lyrics", lines=6, interactive=False)
                    fmeta_out = gr.Textbox(label="📊 Detected Metadata", interactive=False)
                    fbtn.click(format_ui, [fcap, flyr], [fcap_out, flyr_out, fmeta_out])

        gr.HTML('<div class="footer"><p style="font-size: 16px; margin: 5px 0;">🎵 Created by <strong>AIQuest Academy</strong></p><p style="font-size: 14px; margin: 5px 0; color: #9ca3af;">Free & Open Source | MIT License | ACE-Step 1.5 (0.6B Model)</p><p style="font-size: 13px; margin: 10px 0;"><a href="https://youtube.com/@aiquestacademy" target="_blank" style="color: #667eea; text-decoration: none; margin: 0 10px;">YouTube</a> | <a href="https://x.com/aiquestacademy" target="_blank" style="color: #667eea; text-decoration: none; margin: 0 10px;">X (Twitter)</a></p></div>')

        init_btn.click(wrap_init_engine, outputs=[init_status, btn, init_btn]).then(lambda: [gr.update(interactive=True)]*3, outputs=[cbtn, rbtn, fbtn])

    return demo

import acestep
found = False
target_func = "create_gradio_interface"
path = getattr(acestep, "__path__", [])
for _, name, _ in pkgutil.walk_packages(path, prefix="acestep."):
    try:
        mod = importlib.import_module(name)
        if hasattr(mod, target_func):
            print(f"🎯 Patching {target_func} in {mod.__name__}...")
            def wrapper(*args, **kwargs):
                dit = next((a for a in args if "AceStepHandler" in str(type(a))), args[0] if args else None)
                llm = next((a for a in args if "LLMHandler" in str(type(a))), args[1] if len(args) > 1 else None)
                return run_custom_ui(dit, llm, **kwargs)
            setattr(mod, target_func, wrapper)
            found = True
            break
    except ImportError: pass

if not found:
    print("⚠️ Fallback Patch...")
    try:
        import acestep.gradio_ui.interface
        acestep.gradio_ui.interface.create_gradio_interface = lambda *a, **k: run_custom_ui(a[0], a[1], **k)
        print("🎯 Fallback successful!")
    except: print("❌ Failed to patch UI.")

try:
    import tomllib
except ImportError:
    import tomli as tomllib

with open("pyproject.toml", "rb") as f:
    pyproject = tomllib.load(f)

# DO NOT pass --lm_model_path here! It causes model download before Gradio launches.
# The custom UI's wrap_init_engine() handles 0.6B download + init when user clicks "Initialize Engine".
sys.argv = ["acestep", "--share"]

scripts = pyproject.get("project", {}).get("scripts", {})
entry = scripts.get("acestep", "")

if entry and ":" in entry:
    mod_path, func = entry.rsplit(":", 1)
    print(f"🚀 Running CLI: {mod_path}:{func}")
    m = importlib.import_module(mod_path)
    getattr(m, func)()
else:
    import runpy
    runpy.run_module("acestep", run_name="__main__")
'''

with open("/content/ACE-Step-1.5/custom_launch.py", "w") as f:
    f.write(LAUNCHER)
print("📝 Custom launcher written")
print("🎵 Launching AIQuest Academy Music Studio...")
print("🔗 Watch for the public Gradio URL!\n")

!MPLBACKEND=agg uv run python custom_launch.py

/content/ACE-Step-1.5
📝 Custom launcher written
🎵 Launching AIQuest Academy Music Studio...
🔗 Watch for the public Gradio URL!

Loaded configuration from /content/ACE-Step-1.5/.env.example (fallback)
2026-04-25 20:30:05.554 | DEBUG    | acestep.core.generation.handler.init_service_memory_basic:_apply_malloc_mmap_threshold:50 - [memory] Set M_MMAP_THRESHOLD=131072 for immediate OS reclaim of large frees
2026-04-25 20:30:11.767 | WARNING  | acestep.training.trainer:<module>:40 - bitsandbytes not installed. Using standard AdamW.
🎯 Patching create_gradio_interface in acestep.acestep_v15_pipeline...
🚀 Running CLI: acestep.acestep_v15_pipeline:main
2026-04-25 20:30:13.902 | INFO     | acestep.gpu_config:get_gpu_memory_gb:578 - CUDA GPU detected: Tesla T4 (14.6 GB)

GPU Configuration Detected:
  GPU Memory: 14.56 GB
  Configuration Tier: tier5
  Max Duration (with LM): 480s (8 min)
  Max Duration (without LM): 600s (10 min)
  Max Batch Size (with LM): 4
  Max Batch Size (without LM): 4
  Defa

---

<div align="center">

  <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1">
    <img src="https://img.shields.io/badge/Subscribe%20on%20YouTube-FF0000?style=for-the-badge&logo=youtube&logoColor=white" />
  </a>
  <a href="https://x.com/aiquestacademy">
    <img src="https://img.shields.io/badge/Follow%20on%20X-000000?style=for-the-badge&logo=x&logoColor=white" />
  </a>
  <a href="https://aiquest.site">
    <img src="https://img.shields.io/badge/Support%20My%20Work-f59e0b?style=for-the-badge&logoColor=white" />
  </a>

</div>

<p align="center" style="color:#6b7280; font-size:12px; margin-top:8px;">
  ⚡ Made with ❤️ by <strong>AIQUEST Academy</strong> · aiquest.site · © All rights reserved
</p>

---